# Reporte de Práctica 4: Codificación Huffman
En esta práctica se abordó la aplicación de la Codificación Huffman, un algoritmo de compresión de datos que reduce el tamaño de información al asignar códigos de longitud variable a distintos caracteres basado en su frecuencia de aparición.

El algoritmo termina asignando códigos más cortos a los caracteres con la mayor frecuencia, con el fin de ahorrar espacio con los caracteres más comunes y disminuir el tamaño del archivo en la mayor medida posible.

Planteamiento del problema

El problema se divide en 5 pasos y una verificación:
- Leer el archivo de texto a codificar.
- Calcular la frecuencia de cada caractér.
- Construir el árbol de Huffman.
- Generar la tabla de códigos.
- Codificar el texto.
- Verificar que la codificación coincida con el texto.


In [2]:
from collections import Counter
import heapq

ruta = "DiarioTest.txt"

with open(ruta, 'r', encoding='latin-1') as MensajePath:
    text = MensajePath.read()

In [3]:
print("Texto original:\n")
print(text)

Texto original:

Diario de valores
Profesora: Martha Margarita Cordero Dorantes 
Alumno: Martinez Redon Abraham | Grupo: 2AV1
Miércoles 12
Honestidad. Para mi, la honestidad significa decir la verdad, incluso cuando te pueda perjudicar, en mi vida diaria intento aplicarlo siendo franco con las cosas que quiero y no quiero. El mayor obstáculo creo que son las consecuencias que te pueda traer decir la verdad, a veces castigo, a veces un evento indeseable; frecuentemente digo mentiras blancas, mentiras que se dicen por proteger la integridad, ya sea una opinión u algo que simplemente preferiría que no sepan.
El día de hoy, la practiqué admitiendo a mi mamá que no tenía ganas de ir al gimnasio, por lo que ella fue sola.
La honestidad es un valor que no necesariamente me cuesta, no soy un mentiroso compulsivo, pero es cierto que me guardo varios pensamientos o no digo verdades completas; últimamente he tenido también que ser honesto conmigo mismo acerca de cómo me hace sentir la interacción

2.	Calcular la frecuencia de cada caractér.
Se utilizó la función Counter de la librería collections para facilitar el conteo de caracteres, ya que nos devuelve un arreglo con los caracteres y sus respectivas frecuencias.


In [4]:
def calcular_frecuencias(text):
    return Counter(text) ## Funcón de calcular caracteres.

with open(ruta, 'r', encoding='latin-1') as MensajePath:
    text = MensajePath.read()

frecuencias = calcular_frecuencias(text)
print("Frecuencias por caractér:")
for char, freq in frecuencias.items():
    print(f"'{char}': {freq}")


Frecuencias por caractér:
'D': 7
'i': 973
'a': 1712
'r': 928
'o': 1324
' ': 3297
'd': 690
'e': 2164
'v': 171
'l': 746
's': 1066
'
': 123
'P': 6
'f': 118
':': 4
'M': 20
't': 679
'h': 98
'g': 157
'C': 5
'n': 1064
'A': 10
'u': 666
'm': 545
'z': 24
'R': 4
'b': 153
'|': 1
'G': 2
'p': 450
'2': 15
'V': 6
'1': 16
'é': 51
'c': 664
'H': 13
'.': 88
',': 214
'j': 52
'q': 250
'y': 171
'E': 19
'á': 87
';': 20
'ó': 46
'í': 89
'L': 15
'ú': 10
'J': 8
'3': 6
'x': 21
'(': 7
')': 7
'4': 5
'S': 9
'7': 3
'ñ': 19
'5': 4
'6': 2
'': 5
'': 5
'U': 4
'8': 1
'9': 1
'0': 2
'T': 2
'/': 2
'w': 3
'Y': 1
'N': 5
'[': 1
']': 1
'I': 1
'F': 1
'	': 1


3.	Construir el árbol de Huffman.
Para esta sección se utilizó la ayuda de la IA incluida en Google colab, Flash Gemini 2.5, utilizando el siguiente prompt: 
“Se quiere construir un bloque de código que construya el árbol utilizado en la codificación Huffman con min_heap, y que se construya la tabla de códigos por carácter, ordenados por frecuencia de manera descendente.”
De manera que se creó la estructura del árbol, con sus respectivos punteros, y 2 variables que referencíen el caractér y su frecuencia.

In [5]:
# Node class for Huffman tree
class Node:
    def __init__(self, char, freq, left=None, right=None): ## Nodo de un montículo, apuntador, caractér, frecuencia, izquierda, derecha.
        self.char = char
        self.freq = freq
        self.left = left
        self.right = right

    # Usando heapq para comparar basado en frecuencias.
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(frequencies):
    priority_queue = []
    for char, freq in frequencies.items():
        heapq.heappush(priority_queue, Node(char, freq))

    while len(priority_queue) > 1:
        left = heapq.heappop(priority_queue)
        right = heapq.heappop(priority_queue)

        # Crear nuevo nodo con la suma de los 2 nodos hijos.
        merged = Node(None, left.freq + right.freq, left, right)
        heapq.heappush(priority_queue, merged)

    return priority_queue[0]

huffman_tree_root = build_huffman_tree(frecuencias)
print("Árbol de Huffman construído.")


Árbol de Huffman construído.


4.	Generar la tabla de códigos.
Esta porción del código verifica que el nodo actual no se trate de la raíz del árbol y asigna 0 o 1 dependiendo de la manera en que se recorra el árbol.

In [11]:
def generate_huffman_codes(root, current_code="", codes=None):

    if codes is None:
        codes = {}

    if root is None:
        return

    # Nodo hoja
    if root.char is not None:
        codes[root.char] = current_code
        return codes

    # Izquierda = 0
    generate_huffman_codes(
        root.left,
        current_code + "0",
        codes
    )

    # Derecha = 1
    generate_huffman_codes(
        root.right,
        current_code + "1",
        codes
    )

    return codes


huffman_codes = generate_huffman_codes(huffman_tree_root)

print("Códigos de Huffman ordenados por frecuencia:")

sorted_codes = []

for char, code in huffman_codes.items():

    if char in frecuencias:
        sorted_codes.append(
            (frecuencias[char], char, code)
        )

# Orden descendente por frecuencia
sorted_codes.sort(
    key=lambda x: x[0],
    reverse=True
)

for freq, char, code in sorted_codes:
    print(f"'{char}': {code} (Frecuencia: {freq})")

Códigos de Huffman ordenados por frecuencia:
' ': 111 (Frecuencia: 3297)
'e': 011 (Frecuencia: 2164)
'a': 000 (Frecuencia: 1712)
'o': 1010 (Frecuencia: 1324)
's': 1000 (Frecuencia: 1066)
'n': 0101 (Frecuencia: 1064)
'i': 0011 (Frecuencia: 973)
'r': 0010 (Frecuencia: 928)
'l': 11010 (Frecuencia: 746)
'd': 11000 (Frecuencia: 690)
't': 10111 (Frecuencia: 679)
'u': 10110 (Frecuencia: 666)
'c': 10011 (Frecuencia: 664)
'm': 01001 (Frecuencia: 545)
'p': 110111 (Frecuencia: 450)
'q': 010001 (Frecuencia: 250)
',': 1101101 (Frecuencia: 214)
'y': 1100100 (Frecuencia: 171)
'v': 1100101 (Frecuencia: 171)
'g': 1001011 (Frecuencia: 157)
'b': 1001001 (Frecuencia: 153)
'
': 1001000 (Frecuencia: 123)
'f': 0100000 (Frecuencia: 118)
'h': 11011000 (Frecuencia: 98)
'í': 11001110 (Frecuencia: 89)
'.': 11001101 (Frecuencia: 88)
'á': 11001100 (Frecuencia: 87)
'j': 110110011 (Frecuencia: 52)
'é': 110110010 (Frecuencia: 51)
'ó': 110011110 (Frecuencia: 46)
'z': 1100111111 (Frecuencia: 24)
'x': 1001010111 (Frecuen

5.	Codificar el texto.
Por último, se codifica el texto; por cada caractér en el texto se llama al arreglo con los códigos para codificar el texto caractér a caractér.


In [12]:
def encode_text(text, huffman_codes):
    encoded_text = ""
    for char in text:
        encoded_text += huffman_codes[char]
    return encoded_text

encoded_message = encode_text(text, huffman_codes)
print(f"Total de bits codificados: {len(encoded_message)}")
print("Texto codificado (primeros 200 bits):")
print(encoded_message[:200])

Total de bits codificados: 82574
Texto codificado (primeros 200 bits):
01000010011001100000100011101011111000011111110010100011010101000100111000100100011001111100000101010010000001110001010001000001000011100011110010101010000010101111101100000011110010101010000010100101


Verificar que la codificación coincida con el texto.

Con el siguiente prompt se construyó un decodificador:

“Para verificar la correcta codificación, se quiere construir un decodificador de códigos Huffman que trabaje con el texto codificado y el árbol de Huffman propiamente.”

Después, se le agregó una verificación con una variable booleana que será 1 si el texto y la decodificación coinciden.


In [13]:
def decode_text(encoded_text, huffman_tree_root):
    decoded_text = []
    current_node = huffman_tree_root

    for bit in encoded_text:
        if bit == '0':
            current_node = current_node.left
        else: # bit == '1'
            current_node = current_node.right

        if current_node.char is not None: # It's a leaf node
            decoded_text.append(current_node.char)
            current_node = huffman_tree_root # Reset to root for next character

    return "".join(decoded_text)

decoded_message = decode_text(encoded_message, huffman_tree_root)

print("\nTexto decodificado (primeros 200 caracteres):")
print(decoded_message[:200]) # Print only first 200 characters for brevity

# Verification
is_matching = (text == decoded_message)
print(f"\n¿El texto coincide con la codificación? {is_matching}")



Texto decodificado (primeros 200 caracteres):
Diario de valores
Profesora: Martha Margarita Cordero Dorantes 
Alumno: Martinez Redon Abraham | Grupo: 2AV1
Miércoles 12
Honestidad. Para mi, la honestidad significa decir la verdad, incluso cuando t

¿El texto coincide con la codificación? True


## Análisis de resultados

Utilizamos 1 archivos de texto para probar este algoritmo, DiarioTest

Para DiarioTest:

- Tamaño original: 19,283 bytes
- Tamaño codificado: 82574 bits
